# Accelerating Python on CPU: Numba, Dask, and Cython

## Section 1 — Introduction: Why Python is Slow

### Key Technical Concepts
- **The GIL (Global Interpreter Lock):** CPython executes one thread at a time — CPU-bound threads don't benefit from `threading`
- **Dynamic typing overhead:** Every operation involves type checks, object allocation, and reference counting
- **Interpreted bytecode vs. native machine code:** CPython interprets `.pyc` bytecode; C runs directly on silicon
- **Memory layout:** Python lists are arrays of *pointers* to heap objects; NumPy arrays are contiguous memory blocks
- **The acceleration spectrum:**
  - `NumPy` — vectorized operations (already C under the hood)
  - `Numba` — JIT-compile Python/NumPy loops to LLVM IR → machine code
  - `Dask` — parallelize across cores/machines using task graphs
  - `Cython` — transpile annotated Python to C extension modules
  - `C/C++ extensions` — full manual control (ctypes, cffi, pybind11)

In [ ]:
# Setup: install required packages if needed
# !pip install numba dask[dataframe] cython pandas numpy matplotlib

import time
from functools import wraps

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# Utility: timing decorator
def timeit(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"  [{func.__name__}] elapsed: {elapsed:.4f}s")
        return result, elapsed

    return wrapper


print("Environment ready.")

In [ ]:
# Baseline demo: pure Python loop vs. NumPy vectorization
N = 10_000_000
data = list(range(N))
data_np = np.arange(N, dtype=np.float64)


@timeit
def pure_python_sum(arr):
    total = 0.0
    for x in arr:
        total += x * x
    return total


@timeit
def numpy_sum(arr):
    return np.sum(arr * arr)


_, t_python = pure_python_sum(data)
_, t_numpy = numpy_sum(data_np)

print(f"\nSpeedup (NumPy vs pure Python): {t_python / t_numpy:.1f}x")

---
## Section 2 — Numba: JIT Compilation

### Key Technical Concepts
- **JIT = Just-In-Time compilation:** Code is compiled to machine code *at runtime*, the first time a function is called
- **LLVM backend:** Numba uses the same compiler infrastructure as Clang/Swift — production-grade optimization
- **`@jit` vs `@njit`:** `@njit` (= `nopython=True`) forces fully compiled mode — no Python fallback, fastest results
- **Type inference:** Numba infers types from the first call's argument types; it specializes the compiled function for those types
- **Compilation cache:** First call includes compilation overhead (~0.1–2s); subsequent calls hit the compiled version
- **Parallel loops:** `@njit(parallel=True)` + `prange` automatically parallelizes loops across CPU cores
- **When Numba shines:** Tight numerical loops, custom reductions, explicit iteration over arrays
- **When Numba struggles:** String/dict-heavy code, arbitrary Python objects, pandas DataFrames directly

### Resources
1. [Official Numba documentation](https://numba.readthedocs.io/en/stable/user/5minguide.html) — 5-minute guide
2. [Numba: Tell Me When to Use It](https://snarky.ca/what-is-the-core-of-the-python-programming-language/) — practical decision guide
3. [Numba parallel tutorial (GitHub)](https://github.com/ContinuumIO/gtc2018-numba) — GTC 2018 workshop notebooks

In [ ]:
from numba import njit, prange

# ------------------------------------------------------------------
# DEMO 1: Pure Python vs. NumPy vs. Numba — pairwise distance sum
# ------------------------------------------------------------------
# Problem: compute sum of squared distances between consecutive elements

N = 500_000_000
arr = np.random.rand(N).astype(np.float64)


# --- Version 1: Pure Python ---
@timeit
def python_loop(a):
    total = 0.0
    for i in range(len(a) - 1):
        diff = a[i + 1] - a[i]
        total += diff * diff
    return total


# --- Version 2: NumPy vectorized ---
@timeit
def numpy_loop(a):
    diff = np.diff(a)
    return np.sum(diff * diff)


# --- Version 3: Numba JIT (single-threaded) ---
@njit
def _numba_loop_inner(a):
    total = 0.0
    for i in range(len(a) - 1):
        diff = a[i + 1] - a[i]
        total += diff * diff
    return total


@timeit
def numba_loop(a):
    return _numba_loop_inner(a)


# --- Version 4: Numba JIT (parallel) ---
@njit(parallel=True)
def _numba_parallel_inner(a):
    total = 0.0
    for i in prange(len(a) - 1):
        diff = a[i + 1] - a[i]
        total += diff * diff
    return total


@timeit
def numba_parallel(a):
    return _numba_parallel_inner(a)


print("=" * 50)
print("Warming up Numba (first-call compilation)...")
_numba_loop_inner(arr[:100])  # warm up
_numba_parallel_inner(arr[:100])  # warm up
print("Warm-up done. Running benchmarks...\n")

_, t_py = python_loop(arr)
_, t_np = numpy_loop(arr)
_, t_nb = numba_loop(arr)
_, t_nbp = numba_parallel(arr)

print(f"\nSpeedups vs pure Python:")
print(f"  NumPy:          {t_py / t_np:6.1f}x")
print(f"  Numba (serial): {t_py / t_nb:6.1f}x")
print(f"  Numba (parallel):{t_py / t_nbp:5.1f}x")

In [ ]:
# ------------------------------------------------------------------
# DEMO 2: Numba — Monte Carlo Pi estimation
# A classic embarrassingly-parallel workload
# ------------------------------------------------------------------


@timeit
def monte_carlo_pi_python(n_samples):
    import random

    inside = 0
    for _ in range(n_samples):
        x, y = random.random(), random.random()
        if x * x + y * y <= 1.0:
            inside += 1
    return 4.0 * inside / n_samples


@njit(parallel=True)
def _mc_pi_numba(n_samples):
    inside = 0
    for _ in prange(n_samples):
        x = np.random.random()
        y = np.random.random()
        if x * x + y * y <= 1.0:
            inside += 1
    return 4.0 * inside / n_samples


@timeit
def monte_carlo_pi_numba(n_samples):
    return _mc_pi_numba(n_samples)


N_MC = 10_000_000
_mc_pi_numba(1000)  # warm up

print("Monte Carlo Pi Estimation:")
pi_py, t_py_mc = monte_carlo_pi_python(N_MC)
pi_nb, t_nb_mc = monte_carlo_pi_numba(N_MC)

print(f"\n  Python result: π ≈ {pi_py:.5f}")
print(f"  Numba result:  π ≈ {pi_nb:.5f}")
print(f"  Speedup: {t_py_mc / t_nb_mc:.1f}x")

---
## Section 3 — Dask: Parallel Computing Across CPU Cores

### Key Technical Concepts
- **Task graphs:** Dask represents computations as a directed acyclic graph (DAG) of tasks — nothing executes until `.compute()` is called
- **Lazy evaluation:** Build the full computation plan first, then execute optimally (fuses operations, avoids redundant passes)
- **`dask.array` vs NumPy:** Same API, but arrays are chunked — each chunk is a NumPy array processed in parallel
- **`dask.dataframe` vs Pandas:** Same API, but the DataFrame is partitioned row-wise — each partition is a Pandas DataFrame
- **Schedulers:**
  - `synchronous` — single-threaded, useful for debugging
  - `threads` — thread pool, best for I/O-bound tasks (GIL released by NumPy)
  - `processes` — process pool, best for pure Python CPU-bound tasks (bypasses GIL)
  - `distributed` — full cluster scheduler (can run locally too)
- **When Dask shines:** Data larger than RAM, embarrassingly parallel pipelines, scaling existing NumPy/Pandas code
- **When Dask adds overhead:** Small data (task graph overhead > compute time), highly sequential algorithms

### Practical Scenario
Processing a 10M-row dataset: feature engineering pipeline that would OOM or take minutes in Pandas

### Resources
1. [Dask Tutorial (official)](https://tutorial.dask.org/) — hands-on notebooks covering arrays, dataframes, delayed
2. [Dask Best Practices](https://docs.dask.org/en/stable/best-practices.html) — when to use and when not to
3. [Dask for ML Practitioners (Coiled blog)](https://coiled.io/blog/dask-tutorial-for-machine-learning/) — ML-focused case study

In [ ]:
import dask.array as da
import dask.dataframe as dd
from dask import compute, delayed

# ------------------------------------------------------------------
# DEMO 3a: dask.array vs NumPy — large array computation
# ------------------------------------------------------------------

shape = (10_000, 10_000)  # 100M elements = ~800 MB float64


# NumPy: allocates the full array in memory
@timeit
def numpy_large_compute():
    x = np.random.random(shape)
    return np.mean(np.sin(x) ** 2 + np.cos(x) ** 2)


# Dask: chunked, parallel execution
@timeit
def dask_large_compute():
    x = da.random.random(shape, chunks=(1000, 10_000))  # 10 chunks
    result = da.mean(da.sin(x) ** 2 + da.cos(x) ** 2)
    return result.compute()


print("Large Array Computation (sin²+cos² mean):")
_, t_np_arr = numpy_large_compute()
_, t_da_arr = dask_large_compute()

print(f"\n  NumPy elapsed: {t_np_arr:.3f}s")
print(f"  Dask elapsed:  {t_da_arr:.3f}s")
print(f"  Speedup: {t_np_arr / t_da_arr:.2f}x")
print("  (Dask benefits scale with chunk parallelism and available cores)")

In [ ]:
# ------------------------------------------------------------------
# DEMO 3b: dask.dataframe vs Pandas — large dataset feature engineering
# ------------------------------------------------------------------

print("Generating synthetic dataset (5M rows)...")
N_ROWS = 5_000_000
df_pandas = pd.DataFrame(
    {
        "user_id": np.random.randint(0, 100_000, N_ROWS),
        "value": np.random.randn(N_ROWS),
        "category": np.random.choice(["A", "B", "C", "D"], N_ROWS),
    }
)


def feature_pipeline_pandas(df):
    """Feature engineering: groupby aggregation + normalization"""
    agg = df.groupby("user_id")["value"].agg(["mean", "std", "count"]).reset_index()
    agg.columns = ["user_id", "mean_val", "std_val", "count"]
    agg["normalized"] = (agg["mean_val"] - agg["mean_val"].mean()) / (
        agg["std_val"].std() + 1e-9
    )
    return agg


@timeit
def run_pandas():
    return feature_pipeline_pandas(df_pandas)


@timeit
def run_dask():
    ddf = dd.from_pandas(df_pandas, npartitions=8)
    agg = ddf.groupby("user_id")["value"].agg(["mean", "std", "count"]).reset_index()
    agg.columns = ["user_id", "mean_val", "std_val", "count"]
    return agg.compute()


print("\nFeature Engineering Pipeline (groupby + aggregation):")
_, t_pd = run_pandas()
_, t_dd = run_dask()

print(f"\n  Pandas elapsed: {t_pd:.3f}s")
print(f"  Dask elapsed:   {t_dd:.3f}s")
print(f"  Speedup: {t_pd / t_dd:.2f}x")

In [ ]:
# ------------------------------------------------------------------
# DEMO 3c: dask.delayed — parallelizing arbitrary Python functions
# ------------------------------------------------------------------
import time as _time


def slow_transform(x, delay=0.1):
    """Simulates an I/O-bound or CPU-bound task (e.g., loading a file, model inference)"""
    _time.sleep(delay)
    return x**2


items = list(range(200))


@timeit
def sequential_processing():
    return [slow_transform(x, delay=0.05) for x in items]


@timeit
def dask_delayed_processing():
    tasks = [delayed(slow_transform)(x, delay=0.05) for x in items]
    return compute(*tasks)


print("Parallel task graph with dask.delayed (20 tasks, 50ms each):")
_, t_seq = sequential_processing()
_, t_del = dask_delayed_processing()

print(f"\n  Sequential: {t_seq:.3f}s")
print(f"  Dask parallel: {t_del:.3f}s")
print(f"  Speedup: {t_seq / t_del:.2f}x")

---
## Section 4 — Cython: C-Extensions from Python

### Key Technical Concepts
- **What Cython is:** A superset of Python that compiles to C — you write `.pyx` files, Cython generates `.c`, which is compiled to a `.so` extension
- **The acceleration model:** Every `cdef` typed variable removes a Python object allocation; every `cpdef` function removes Python call overhead
- **Static type declarations:** `cdef int i`, `cdef double[:] arr` (typed memory views) — these are the key to C-speed
- **`cdef` vs `cpdef` vs `def`:**
  - `def` — callable from Python, Python overhead
  - `cdef` — C-only, not callable from Python, fastest
  - `cpdef` — callable from both, slight overhead vs pure `cdef`
- **Typed memory views:** Direct buffer access to NumPy arrays without copying — `double[::1]` is contiguous C array
- **When to prefer Cython over Numba:**
  - Wrapping existing C/C++ libraries
  - Complex control flow Numba can't infer
  - Shipping compiled extensions (no JIT overhead)
  - Fine-grained memory control
  - Integration with CPython internals

### Workflow: Converting a Python Module to Cython
1. Rename `module.py` → `module.pyx`
2. Add `cdef` type annotations to hot variables and loop indices
3. Use typed memory views for NumPy array arguments
4. Create `setup.py` with `cythonize()` and run `python setup.py build_ext --inplace`
5. Profile with `cython -a module.pyx` — yellow lines = Python overhead remaining

### Resources
1. [Cython Official Tutorial](https://cython.readthedocs.io/en/latest/src/tutorial/cython_tutorial.html) — basic tutorial
2. [Working with NumPy in Cython](https://cython.readthedocs.io/en/latest/src/tutorial/numpy.html) — typed memoryviews guide
3. [Fast Numerical Computations with Cython (SciPy Lecture Notes)](https://scipy-lectures.org/advanced/optimizing/index.html#cython) — hands-on examples

In [ ]:
# ------------------------------------------------------------------
# DEMO 4a: Cython via IPython magic (%% cython)
# Requires: pip install cython && jupyter nbextension enable cython
# ------------------------------------------------------------------
# Load Cython magic
%load_ext Cython

In [ ]:
%%cython
# cython: boundscheck=False, wraparound=False, cdivision=True
import numpy as np
cimport numpy as cnp

def pairwise_dist_cython(double[::1] a):
    """
    Cython version: sum of squared consecutive differences.
    Same algorithm as the pure Python and Numba versions.
    """
    cdef:
        Py_ssize_t i, n = len(a)
        double total = 0.0
        double diff

    for i in range(n - 1):
        diff = a[i+1] - a[i]
        total += diff * diff

    return total

In [ ]:
# Now benchmark the Cython version against previous approaches
N = 50_000_000
arr = np.random.rand(N).astype(np.float64)


@timeit
def cython_pairwise(a):
    return pairwise_dist_cython(a)


print("Pairwise distance sum — Cython vs others:")
_, t_py_c = python_loop(arr)
_, t_np_c = numpy_loop(arr)
_, t_nb_c = numba_loop(arr)
_, t_cy = cython_pairwise(arr)

print(f"\n  Pure Python: {t_py_c:.4f}s  (1.0x baseline)")
print(f"  NumPy:       {t_np_c:.4f}s  ({t_py_c / t_np_c:.1f}x)")
print(f"  Numba:       {t_nb_c:.4f}s  ({t_py_c / t_nb_c:.1f}x)")
print(f"  Cython:      {t_cy:.4f}s   ({t_py_c / t_cy:.1f}x)")

In [ ]:
%%cython
# cython: boundscheck=False, wraparound=False, cdivision=True
# DEMO 4b: Monte Carlo Pi in Cython for direct comparison
from libc.stdlib cimport rand, RAND_MAX

def monte_carlo_pi_cython(long n_samples):
    cdef:
        long i, inside = 0
        double x, y

    for i in range(n_samples):
        x = rand() / <double>RAND_MAX
        y = rand() / <double>RAND_MAX
        if x*x + y*y <= 1.0:
            inside += 1

    return 4.0 * inside / n_samples

In [ ]:
@timeit
def run_mc_cython(n):
    return monte_carlo_pi_cython(n)


N_MC = 100_000_000
print("Monte Carlo Pi — Cython vs others:")
pi_py_c, t_py_mc2 = monte_carlo_pi_python(N_MC)
pi_nb_c, t_nb_mc2 = monte_carlo_pi_numba(N_MC)
pi_cy, t_cy_mc = run_mc_cython(N_MC)

print(f"\n  Pure Python: π ≈ {pi_py_c:.5f}  time: {t_py_mc2:.4f}s (1.0x)")
print(
    f"  Numba:       π ≈ {pi_nb_c:.5f}  time: {t_nb_mc2:.4f}s ({t_py_mc2 / t_nb_mc2:.1f}x)"
)
print(
    f"  Cython:      π ≈ {pi_cy:.5f}  time: {t_cy_mc:.4f}s ({t_py_mc2 / t_cy_mc:.1f}x)"
)

---
## Section 5 — Head-to-Head Benchmark Summary

A unified visual comparison across all three tools and workloads.

In [ ]:
# ------------------------------------------------------------------
# COMPREHENSIVE BENCHMARK: All approaches, two workloads
# ------------------------------------------------------------------

import warnings

warnings.filterwarnings("ignore")

N_BENCH = 5_000_000
N_MC_BENCH = 10_000_000
arr_bench = np.random.rand(N_BENCH).astype(np.float64)

RUNS = 3  # average over multiple runs for stability


def avg_time(fn, *args, runs=RUNS):
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn(*args)
        times.append(time.perf_counter() - t0)
    return min(times)  # best-of to reduce noise


print("Running benchmarks (best of 3 runs)...")

# --- Workload A: Pairwise distance sum ---
t_pw = {
    "Pure Python": avg_time(
        lambda a: sum((a[i + 1] - a[i]) ** 2 for i in range(len(a) - 1)),
        arr_bench[:100_000],
    ),
    "NumPy": avg_time(lambda a: np.sum(np.diff(a) ** 2), arr_bench),
    "Numba (serial)": avg_time(_numba_loop_inner, arr_bench),
    "Numba (parallel)": avg_time(_numba_parallel_inner, arr_bench),
    "Cython": avg_time(pairwise_dist_cython, arr_bench),
}
# Scale Python time to full N (since we sampled 100k)
t_pw["Pure Python"] *= N_BENCH / 100_000

# --- Workload B: Monte Carlo Pi ---
t_mc = {
    "Pure Python": avg_time(
        lambda n: (
            __import__("random")
            and sum(
                1
                for _ in range(n)
                if (
                    __import__("random").random() ** 2
                    + __import__("random").random() ** 2
                )
                <= 1
            )
        ),
        500_000,
    ),
    "Numba (parallel)": avg_time(_mc_pi_numba, N_MC_BENCH),
    "Cython": avg_time(monte_carlo_pi_cython, N_MC_BENCH),
}
t_mc["Pure Python"] *= N_MC_BENCH / 500_000

print("Done!")

In [ ]:
# ------------------------------------------------------------------
# Visualization: Throughput (1/time) bar charts — higher = faster
# ------------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    "Python CPU Acceleration: Head-to-Head Throughput (higher = faster)",
    fontsize=14,
    fontweight="bold",
)

colors = {
    "Pure Python": "#e74c3c",
    "NumPy": "#3c83b2",
    "Numba (serial)": "#6717d7",
    "Numba (parallel)": "#27ae60",
    "Cython": "#c2dc1b",
    "Dask": "#f39c12",
}

# -- Plot A: Pairwise distance sum --
ax = axes[0]
labels_a = list(t_pw.keys())
throughput_a = [1.0 / t for t in t_pw.values()]  # 1/s — higher = faster
baseline_a = 1.0 / t_pw["Pure Python"]
speedup_a = [tp / baseline_a for tp in throughput_a]

bars = ax.bar(
    labels_a,
    throughput_a,
    color=[colors.get(l, "#95a5a6") for l in labels_a],
    edgecolor="white",
    linewidth=0.8,
)
ax.set_title("Workload A: Pairwise Distance Sum\n(5M elements)", fontsize=12)
ax.set_ylabel("Throughput  (1 / seconds)  ↑ higher is faster")
ax.set_xticklabels(labels_a, rotation=20, ha="right")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1f}"))

for bar, sp in zip(bars, speedup_a):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() * 1.02,
        f"{sp:.1f}x",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
    )

# -- Plot B: Monte Carlo Pi --
ax = axes[1]
labels_b = list(t_mc.keys())
throughput_b = [1.0 / t for t in t_mc.values()]
baseline_b = 1.0 / t_mc["Pure Python"]
speedup_b = [tp / baseline_b for tp in throughput_b]

bars = ax.bar(
    labels_b,
    throughput_b,
    color=[colors.get(l, "#95a5a6") for l in labels_b],
    edgecolor="white",
    linewidth=0.8,
)
ax.set_title("Workload B: Monte Carlo π Estimation\n(10M samples)", fontsize=12)
ax.set_ylabel("Throughput  (1 / seconds)  ↑ higher is faster")
ax.set_xticklabels(labels_b, rotation=20, ha="right")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1f}"))

for bar, sp in zip(bars, speedup_b):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() * 1.02,
        f"{sp:.1f}x",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
    )

plt.tight_layout()
plt.savefig("benchmark_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to benchmark_comparison.png")

In [ ]:
# ------------------------------------------------------------------
# Speedup summary table
# ------------------------------------------------------------------

print("=" * 55)
print(" SPEEDUP SUMMARY (vs. Pure Python baseline)")
print("=" * 55)
print(f"{'Approach':<22} {'Workload A':>12} {'Workload B':>12}")
print("-" * 55)

all_labels = list(set(list(t_pw.keys()) + list(t_mc.keys())))
for label in ["Pure Python", "NumPy", "Numba (serial)", "Numba (parallel)", "Cython"]:
    sp_a = f"{t_pw['Pure Python'] / t_pw[label]:.1f}x" if label in t_pw else "N/A"
    sp_b = f"{t_mc['Pure Python'] / t_mc[label]:.1f}x" if label in t_mc else "N/A"
    print(f"  {label:<20} {sp_a:>12} {sp_b:>12}")

print("=" * 55)
print("  Note: Dask speedup is workload/data-size dependent")
print("  and measured separately on the dataframe benchmark.")

---
## Section 6 — Tool Selection Framework

### Decision Flowchart

```
Is your bottleneck a tight numerical loop over arrays?
    YES → Try Numba @njit first (zero code rewrite required)
          ├── Works? → Done. Add parallel=True for extra speedup.
          └── Numba can't handle it (strings, dicts, objects)? → Cython

Is your bottleneck data processing at scale (large files, many partitions)?
    YES → Use Dask (drop-in for Pandas/NumPy)
          ├── Data fits in RAM, just want multi-core? → Dask with local threads/processes
          └── Data exceeds RAM? → Dask with chunked arrays/dataframes

Do you need to wrap a C/C++ library or ship a compiled extension?
    YES → Cython (or pybind11 for C++)

Do you need fine-grained memory control or CPython internals?
    YES → Cython
```

### Takeaway Cheat Sheet

| Criterion | Numba | Dask | Cython |
|-----------|-------|------|--------|
| **Best for** | Numerical loops, array math | Large data, multi-core pipelines | C interop, compiled extensions |
| **Code change required** | Minimal (decorator) | Minimal (same API) | Moderate (type annotations) |
| **Compilation** | JIT at runtime | None | AOT (build step required) |
| **Startup overhead** | ~0.1–2s (first call) | None | None (pre-compiled) |
| **Parallel support** | Yes (`parallel=True`) | Yes (schedulers) | Yes (with OpenMP) |
| **GPU support** | Yes (CUDA) | Limited | No |
| **Handles Pandas** | Partial | Yes (native) | Manual |
| **Handles arbitrary Python** | No (type-restricted) | Yes (delayed) | Partial |
| **Typical speedup** | 10–200x | 2–8x (more cores) | 10–100x |
| **Learning curve** | Low | Low–Medium | Medium–High |
| **Deployment** | pip install numba | pip install dask | Requires build env |

### When NOT to use each tool
- **Numba:** String processing, complex Python objects, I/O-bound tasks
- **Dask:** Small data (overhead > gain), highly sequential algorithms, real-time low-latency
- **Cython:** Rapid prototyping, when Numba already works, teams without C build toolchains

---

## Resource Library

### Numba
1. [5-Minute Numba Guide](https://numba.readthedocs.io/en/stable/user/5minguide.html) — official quickstart
2. [Numba Parallel Workshop Notebooks (GTC 2018)](https://github.com/ContinuumIO/gtc2018-numba) — hands-on labs
3. [Numba Performance Tips](https://numba.readthedocs.io/en/stable/user/performance-tips.html) — official tuning guide

### Dask
1. [Dask Interactive Tutorial](https://tutorial.dask.org/) — official notebooks (binder-ready)
2. [Dask Best Practices](https://docs.dask.org/en/stable/best-practices.html) — when and how to use
3. [Scaling Pandas with Dask (Real Python)](https://realpython.com/pandas-groupby/) — ML practitioner focus

### Cython
1. [Cython Basic Tutorial](https://cython.readthedocs.io/en/latest/src/tutorial/cython_tutorial.html) — official
2. [Cython + NumPy Tutorial](https://cython.readthedocs.io/en/latest/src/tutorial/numpy.html) — typed memoryviews
3. [SciPy Lectures: Optimizing with Cython](https://scipy-lectures.org/advanced/optimizing/index.html#cython) — practical guide

In [ ]:
# ------------------------------------------------------------------
# BONUS: Profile a function to find the actual bottleneck first
# Golden rule: measure before optimizing
# ------------------------------------------------------------------
import cProfile
import io
import pstats


def realistic_pipeline(n=500_000):
    """A mixed workload: loop + array ops + string formatting"""
    arr = np.random.rand(n)
    result = 0.0
    for i in range(0, n, 100):
        result += arr[i]
    labels = [f"item_{i}" for i in range(1000)]
    return result, labels


pr = cProfile.Profile()
pr.enable()
realistic_pipeline()
pr.disable()

s = io.StringIO()
ps = pstats.Stats(pr, stream=s).sort_stats("cumulative")
ps.print_stats(10)
print("Top 10 functions by cumulative time:")
print(s.getvalue())
print("=> Identify the hot spot, THEN choose your tool.")

---

## Summary

| Step | Action |
|------|--------|
| 1 | **Profile first** with `cProfile` / `line_profiler` — find the real bottleneck |
| 2 | **Vectorize** with NumPy if not already done |
| 3 | **Add `@njit`** (Numba) to remaining tight loops |
| 4 | **Add `parallel=True`** and `prange` to use all cores |
| 5 | **Switch to Dask** if data exceeds RAM or pipeline is multi-stage |
| 6 | **Write Cython** for C library integration or deployable extensions |